In [ ]:
!mkdir -p src tests sample/expected
!test -d eigen-3.4.0 || (wget -q https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.tar.gz && tar -xzf eigen-3.4.0.tar.gz)
!test -f doctest.h || wget -q https://raw.githubusercontent.com/doctest/doctest/v2.5.3/doctest/doctest.h
!g++ --version | head -1

g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


In [ ]:
# @title check_outputs.py: python program to compare your summary.csv and corr.csv with the expected ones. (to generate the file just run the cell, do not edit any code in it) { display-mode: "form" }
%%writefile check_outputs.py
"""check_outputs.py: compare your summary.csv and corr.csv with the expected ones.

Usage:  python3 check_outputs.py <folder with your outputs> <folder with the expected outputs>
Numbers are compared with a tolerance (1.5e-6, the rounding of 6 decimals), not as text.
Exit status 0 if both files match.
"""
import csv
import math
import os
import sys

TOL = 1.5e-6
FILES = ("summary.csv", "corr.csv")


def rows(path):
    with open(path, newline="") as f:
        return [[c.strip() for c in r] for r in csv.reader(f) if r and any(c.strip() for c in r)]


def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


def compare_table(got, want, header):
    """(ok, message). With header=True the first row must be identical and the first column holds names."""
    if header:
        if got[:1] != want[:1]:
            return False, f"header is {got[:1]} not {want[:1]}"
        got, want = got[1:], want[1:]
    if len(got) != len(want):
        return False, f"{len(got)} rows, expected {len(want)}"
    worst = 0.0
    for i, (g, w) in enumerate(zip(got, want), start=1):
        if len(g) != len(w):
            return False, f"row {i} has {len(g)} fields, expected {len(w)}"
        for j, (x, y) in enumerate(zip(g, w), start=1):
            if header and j == 1 and not is_number(y):      # an asset name
                if x != y:
                    return False, f"row {i}: '{x}' not '{y}'"
                continue
            if not is_number(x):
                return False, f"row {i}, field {j}: '{x}' is not a number"
            xv, yv = float(x), float(y)
            if math.isnan(xv) or math.isinf(xv):
                return False, f"row {i}, field {j}: '{x}'"
            if abs(xv - yv) > TOL:
                return False, f"row {i}, field {j}: {x} but expected {y}"
            worst = max(worst, abs(xv - yv))
    return True, f"largest difference {worst:.1e}"


def compare(out_dir, key_dir):
    """Returns {filename: (ok, message)} for summary.csv and corr.csv."""
    results = {}
    for name in FILES:
        got_path = os.path.join(out_dir, name)
        if not os.path.exists(got_path):
            results[name] = (False, "not written")
            continue
        try:
            results[name] = compare_table(rows(got_path), rows(os.path.join(key_dir, name)), header=(name == "summary.csv"))
        except Exception as e:
            results[name] = (False, f"could not be read: {e}")
    return results


def main(argv):
    if len(argv) != 3:
        print(__doc__)
        return 2
    results = compare(argv[1], argv[2])
    for name, (ok, msg) in results.items():
        print(f"{'PASS' if ok else 'FAIL'}  {name:12s} {msg}")
    return 0 if all(ok for ok, _ in results.values()) else 1


if __name__ == "__main__":
    sys.exit(main(sys.argv))

Overwriting check_outputs.py


In [ ]:
# @title Your data and the sample files (to generate the files with the data just run the cell, do not edit any code in it) { display-mode: "form" }
from pathlib import Path
Path("sample/expected").mkdir(parents=True, exist_ok=True)

Path("prices.csv").write_text("""date,AAA,BBB,CCC,DDD,EEE,FFF,GGG,HHH
2024-01-02,157.050000,258.450000,44.170000,155.940000,82.900000,134.750000,267.320000,223.260000
2024-01-03,158.655971,257.158861,43.778471,153.679489,83.079525,137.074773,262.006500,217.362513
2024-01-04,158.929982,255.879745,43.951038,152.170817,81.148771,139.656874,256.017394,218.841434
2024-01-05,158.856019,254.089189,44.076340,152.566510,80.345659,141.343477,256.621490,218.034073
2024-01-08,159.649513,254.912409,45.223401,155.156306,81.822550,141.308187,258.765663,224.183139
2024-01-09,157.293334,253.409093,44.153888,151.716115,81.977744,141.753536,253.395987,213.205329
2024-01-10,156.672429,251.641132,43.505075,151.562869,79.825011,142.363893,256.516948,209.308002
2024-01-11,155.390151,250.265250,42.594373,153.469297,79.858589,142.524316,268.279776,207.035284
2024-01-12,155.077605,252.340286,42.697545,155.032303,81.517473,139.483185,273.028055,208.679818
2024-01-15,156.023429,248.566917,41.966655,154.372714,80.364816,140.994501,275.284061,208.158973
2024-01-16,156.950303,248.947537,43.306389,157.016428,80.594364,145.703605,276.139510,208.741098
2024-01-17,155.990031,248.826870,-43.155231,156.372375,80.902403,144.463283,274.844728,211.279500
2024-01-18,158.003076,247.738046,42.806490,155.682789,80.413128,142.676608,274.605648,208.261683
2024-01-19,161.209126,247.349175,43.942285,159.237743,80.963366,144.573730,279.903765,209.191315
2024-01-22,161.788235,249.395453,44.081893,162.689483,81.801420,144.585411,291.036100,209.229520
2024-01-23,161.457420,253.740324,44.540324,163.203887,82.496739,142.359925,289.946723,212.203533
2024-01-24,160.506014,255.989143,44.553709,161.011893,81.638957,141.698214,282.278652,214.023395
2024-01-25,161.753253,254.612284,44.553227,162.179855,80.311584,142.917571,286.310555,209.809570
2024-01-26,164.068191,254.977570,44.856436,162.846907,79.950606,143.713095,286.749445,210.552190
2024-01-29,165.090030,257.756530,45.256768,162.206004,79.070065,143.627342,282.122103,208.957152
2024-01-30,163.403437,258.559979,45.628768,160.469055,80.505779,144.369256,274.064905,212.966835
2024-01-31,161.961270,260.443407,45.377371,160.838341,81.811536,144.308801,276.768190,219.509970
2024-02-01,161.593262,257.583533,44.077400,157.510393,81.235924,146.128393,273.174311,211.319776
2024-02-02,163.493465,260.283234,44.549189,160.662477,80.773086,145.477554,281.106170,208.825165
2024-02-05,164.984162,260.210436,45.588405,164.548949,81.558294,145.680935,288.139456,205.741011
2024-02-06,165.956875,260.067831,45.991869,166.272849,82.582380,145.449396,291.609111,200.739837
2024-02-07,166.253323,258.364254,45.961655,166.774413,82.793170,146.143757,293.521669,202.225291
2024-02-08,167.034229,262.578888,46.679784,172.036576,82.297557,143.670920,307.343975,202.196300
2024-02-09,164.430228,263.241123,45.648031,168.366373,80.784019,143.556599,301.014376,196.908481
2024-02-12,165.350536,261.808078,46.218126,170.755574,81.242769,145.439881,305.778828,197.417565
2024-02-14,163.638337,264.603658,45.558379,167.509004,79.326069,144.857718,298.579568,200.650147
2024-02-13,164.745578,264.536182,46.508218,170.104271,80.485728,143.317232,301.677112,200.866107
2024-02-15,163.095953,261.933381,44.813861,169.419034,79.959551,148.123706,310.163260,198.079752
2024-02-16,163.423585,260.730421,45.102737,165.926278,80.541641,149.377610,295.917043,194.145089
2024-02-19,165.258793,262.721228,46.225514,170.656224,80.842230,146.770012,305.383184,193.821312
2024-02-20,164.947478,260.397873,45.636451,168.901301,80.745674,147.773743,302.966319,197.501462
2024-02-21,165.826725,261.466821,45.862179,168.725056,80.460125,149.092335,300.916855,195.322432
2024-02-22,164.540867,261.081480,45.299092,166.120545,79.769244,149.761339,295.361431,195.288563
2024-02-23,163.784814,260.412947,44.404772,166.052037,79.304155,151.504901,300.831131,193.048565
2024-02-26,162.543024,259.998152,42.805260,163.066188,78.837670,150.520407,300.731509,192.025930
2024-02-27,161.435044,259.814326,42.003366,163.886444,77.668187,151.759366,309.273471,191.527012
2024-02-28,162.022174,260.397321,41.470777,163.310312,78.053126,148.813660,310.934129,193.419942
2024-02-29,161.259735,260.148063,40.342758,165.442131,77.124096,149.778939,327.506136,190.490761
2024-03-01,159.882717,262.182683,40.554012,164.548370,76.755376,148.045596,322.423985,197.606933
2024-03-04,161.405578,265.720777,41.137248,166.965298,79.221393,147.061789,327.254821,198.289898
2024-03-05,163.843752,268.412232,42.508663,170.513893,81.367532,145.276261,330.454211,200.928902
2024-03-06,163.183890,266.249876,41.629612,169.887216,81.060368,146.156438,334.730322,193.833288
2024-03-07,162.196804,266.209276,41.561377,169.883180,80.683399,146.416211,335.242689,194.883876
2024-03-08,163.001581,264.874824,41.922140,167.726708,,148.179751,324.245721,192.431599
2024-03-11,162.223741,262.985749,40.142118,166.702873,77.578240,148.804659,334.013849,188.476243
2024-03-12,163.392754,260.294741,39.692484,164.787030,77.781037,151.457974,330.081654,185.924814
2024-03-13,163.744574,255.407260,38.817787,162.099786,78.160766,155.889611,326.526352,180.718977
2024-03-14,162.547084,254.960774,38.672420,160.808868,77.740201,155.310786,322.607927,186.148535
2024-03-15,161.791255,253.973792,38.725582,159.374993,77.617194,155.895915,316.573635,184.396427
2024-03-18,162.062575,252.129985,38.598889,163.434323,77.322372,154.647860,333.626013,183.430585
2024-03-19,162.860515,253.916279,39.746877,166.981402,77.908676,155.012803,338.310000,185.631403
2024-03-20,162.112295,255.252775,39.339989,168.940046,76.195952,153.408829,349.573706,183.533145
2024-03-21,162.335595,253.265792,39.068703,167.891930,76.727103,151.877780,347.637466,182.981469
2024-03-22,161.034451,255.016177,38.800175,167.818422,75.591194,151.432653,349.648367,185.028850
2024-03-25,159.814012,256.685824,38.748970,167.917288,74.967837,150.259864,350.496800,189.085285
2024-03-26,157.092234,255.597901,38.285778,170.613588,74.071787,150.409741,365.819657,188.373942
2024-03-27,159.106941,256.698571,38.430880,173.433598,74.483082,151.306027,376.399163,187.003429
2024-03-28,157.873013,257.961473,38.102289,172.769155,75.909555,149.262689,376.671694,185.395629
2024-03-29,160.075631,259.967445,39.341609,176.513634,75.498437,146.463086,380.948256,180.962075
2024-04-01,158.245216,258.285263,37.695783,168.018036,74.912702,149.580319,360.323766,179.141885
2024-04-02,156.741423,262.718534,37.175712,168.008449,74.794339,146.940117,365.124676,178.293021
2024-04-03,156.919496,259.923131,36.397065,168.115326,74.792506,148.537073,373.091316,176.228398
2024-04-04,158.765095,261.844971,37.207182,175.982569,76.180536,146.326465,399.553953,174.893897
2024-04-05,158.103837,261.077007,36.425543,175.210195,73.972011,147.378317,404.286213,176.232813
2024-04-08,159.986843,262.610915,37.729609,179.153103,74.832054,148.305744,408.284254,183.748902
2024-04-09,162.205459,260.382962,38.298605,178.775495,75.773626,147.523583,400.793027,181.765062
2024-04-10,162.587996,261.713053,38.119918,180.586222,77.265453,147.107983,410.632315,181.449361
2024-04-11,163.120026,265.669928,38.770120,185.025699,79.615872,145.179169,423.718317,182.033255
2024-04-12,161.245583,264.500725,38.285495,180.629834,78.618936,143.576490,409.119566,177.478827
2024-04-15,161.032099,268.208412,39.188501,183.157955,80.186819,138.106058,411.110708,183.381196
2024-04-16,161.820061,268.069397,39.810540,182.329735,80.460346,141.780388,401.354560,182.127011
2024-04-17,161.141474,268.101164,39.757238,185.008902,80.195047,143.679971,413.531942,183.092337
2024-04-18,156.442576,269.642888,39.114487,181.339652,80.111478,141.736870,403.879224,182.320253
2024-04-19,156.943566,272.419458,40.788246,184.502393,80.779976,138.832150,401.320841,182.358779
2024-04-22,158.200096,272.045773,40.823626,186.769107,81.232352,138.904192,410.703163,177.243031
2024-04-23,159.315474,275.377808,41.499557,187.450393,82.506023,137.058018,407.170644,182.149464
2024-04-24,159.596817,277.919693,41.739270,186.417671,84.091249,136.551579,400.561038,178.067134
2024-04-25,160.408189,276.693103,41.999440,184.291470,83.165055,138.029919,389.320761,177.498924
2024-04-26,163.016917,275.119147,43.031712,192.489699,83.537566,138.910759,414.227782,172.071301
2024-04-29,161.557816,279.058761,44.102743,199.039339,85.037750,138.925191,431.990253,171.651901
2024-04-30,162.184098,278.189590,43.745843,195.700444,85.228885,141.426192,421.170826,169.475184
2024-05-01,160.494330,276.328646,43.643017,194.547617,85.729654,140.824482,417.262212,169.228649
2024-05-02,161.557236,280.737012,44.361157,195.343314,87.618738,140.801574,414.070864,166.833407
2024-05-03,159.620688,277.782682,43.219980,188.318671,86.697441,141.838214,395.150478,168.103224
2024-05-06,159.217049,275.864419,42.852276,191.138524,84.058274,141.859006,410.189209,167.051864
2024-05-07,160.094537,276.862395,43.421105,195.932869,85.028525,141.404427,425.185118,162.663358
2024-05-08,159.944663,276.046591,42.720031,193.586959,86.105120,141.410973,421.806121,162.748517
2024-05-09,160.500282,274.118361,42.676096,196.503268,84.625517,143.141293,434.785389,160.889418
2024-05-10,159.370341,273.524391,42.675626,195.587460,84.459334,143.941839,430.825656,161.393494
2024-05-13,158.812662,273.048879,43.433607,196.536887,84.758195,145.590524,427.640723,163.661970
2024-05-14,159.603483,276.635972,44.601203,198.768721,86.882465,144.693912,426.212131,161.095585
2024-05-15,159.775734,279.208626,45.493042,201.219806,85.491662,144.266111,428.352415,162.387617
2024-05-16,162.497060,283.405942,47.833689,213.254268,86.392933,141.782924,457.437227,162.994209
2024-05-17,162.897620,287.856603,48.340024,211.394304,88.507968,139.183275,445.121065,165.389618
2024-05-20,161.995023,289.908636,48.722921,212.671778,90.802052,137.908785,447.009357,167.916213
2024-05-21,162.050820,290.695043,48.668346,213.488556,91.334998,139.304598,450.866997,165.843096
2024-05-22,163.245048,293.278406,49.921263,214.297143,90.698572,138.174384,443.263073,172.590038
2024-05-23,166.544726,293.462002,51.633671,222.129375,91.644373,138.835599,460.422339,169.281971
2024-05-24,167.760216,290.863997,51.595452,223.111404,91.233525,140.423861,464.752549,170.026051
2024-05-27,165.885673,290.511361,50.209398,216.574014,90.631511,143.454294,450.049257,173.983916
2024-05-28,167.716234,294.494326,51.522146,223.805244,90.330093,142.293823,468.229903,176.300332
2024-05-29,167.197425,296.269869,51.102577,218.991239,91.221762,141.877577,452.224747,178.125927
2024-05-30,166.933476,299.497548,51.641391,215.798567,91.690326,142.119819,434.982295,178.229935
2024-05-31,166.182930,299.396229,51.175065,213.505857,91.295350,141.491376,429.695699,174.172097
2024-06-03,167.426998,299.070803,51.464996,214.301177,92.293178,140.368882,430.496753,178.702711
2024-06-04,169.134489,304.956109,53.857322,216.490730,94.655564,137.191969,420.404423,183.192288
2024-06-05,169.811368,307.393780,54.822703,216.308596,94.218434,134.873319,412.607811,183.068989
2024-06-06,166.957773,306.181094,54.059783,216.727203,93.035878,133.639670,419.789222,184.498985
2024-06-07,165.863420,302.855104,52.494431,213.441947,90.720907,136.828940,419.068251,181.706969
2024-06-10,164.313982,302.614745,52.184311,209.482604,90.997931,139.259694,406.267793,182.394186
2024-06-11,164.128036,302.829918,52.425313,207.469741,92.774187,140.091410,396.887881,185.819546
2024-06-12,165.975149,305.112788,52.806185,208.526436,91.959070,141.139087,398.082920,191.544272
2024-06-13,167.386517,306.709647,53.453802,214.262913,93.492327,140.097190,414.951093,194.036295
2024-06-14,167.031829,307.206567,52.043136,211.854047,92.548060,140.526037,416.417184,191.905610
2024-06-17,166.152416,309.930246,52.916118,212.649391,95.231959,137.816043,412.836777,191.630710
2024-06-18,166.430179,306.938068,53.460197,214.240639,97.158051,138.525486,414.818840,192.690323
2024-06-19,166.998280,306.352809,53.855771,215.053844,96.851180,139.302630,414.962184,192.928708
2024-06-20,166.966654,305.035668,53.580808,214.125688,96.656072,138.101696,413.486296,192.866714
2024-06-21,169.256151,307.827138,56.645918,219.220973,97.292937,136.890600,410.464234,196.470512
2024-06-24,167.107011,308.354644,55.356609,214.985794,97.339554,136.747107,403.896332,196.537231
2024-06-25,167.499498,310.757102,56.100566,218.751373,98.702194,135.244688,412.558779,196.620313
2024-06-26,168.372362,309.480076,55.524785,218.515015,99.438563,135.885379,415.786030,193.591867
2024-06-27,170.506827,306.539833,55.010436,220.506233,100.716549,137.746072,427.049155,194.397875
2024-06-28,170.236448,309.299335,54.512130,221.832080,99.230793,137.603897,435.892795,193.883295
2024-07-01,171.677933,306.334722,53.377241,220.749499,98.268833,139.784663,440.547231,190.375877
2024-07-02,173.294346,306.808202,53.550051,219.590581,99.920058,137.332214,434.671706,194.668816
2024-07-03,172.816871,306.652680,53.325388,221.450941,98.583751,137.703358,443.710391,195.522399
2024-07-04,171.476165,303.693635,52.019188,215.248495,97.988366,141.394577,429.790381,195.015149
2024-07-05,170.719224,302.048902,51.867281,217.763174,95.876281,142.262843,440.928103,195.675636
2024-07-08,169.199645,298.673575,50.732504,216.536720,95.127587,142.846246,445.439407,197.308347
2024-07-09,167.967152,298.770984,50.240346,210.312661,95.109747,142.883900,424.639831,196.381916
2024-07-10,172.593282,298.880036,52.026676,213.376816,96.225067,144.596355,422.434699,193.891637
2024-07-11,173.779180,296.682370,52.273582,215.656612,97.422537,144.766487,429.372409,193.107189
2024-07-12,172.189140,294.293203,50.745430,208.357811,96.736966,148.001101,412.947229,190.185264
2024-07-15,172.707577,291.352665,49.895497,202.575058,97.418125,151.369073,397.167957,189.418786
2024-07-16,170.070788,290.236667,48.596303,197.598665,96.612247,152.663909,387.974035,192.580978
2024-07-17,171.744454,291.405208,49.539248,201.588120,96.590436,151.882747,396.100343,194.708540
2024-07-18,168.814680,292.694244,48.719423,199.441326,96.447419,150.614319,394.140501,193.861950
2024-07-19,169.939055,291.007940,49.391903,202.309316,94.548042,153.196077,400.026909,192.999433
2024-07-22,170.658918,291.916818,49.613769,201.061034,94.856235,153.015675,393.503455,192.058654
2024-07-23,171.886926,293.084596,49.220702,203.620473,95.553104,152.708158,406.479944,187.822073
2024-07-24,172.199809,293.271440,49.456319,206.333593,92.236618,151.381724,415.258027,190.401496
2024-07-25,172.111913,292.637576,50.388271,210.419912,93.366382,150.798620,423.865541,190.891280
2024-07-26,172.058579,291.770598,49.579001,211.107114,91.352589,151.469861,433.271956,192.203768
2024-07-29,174.085314,295.390107,51.013790,216.706789,93.104033,150.162517,443.761959,197.053841
2024-07-30,173.308930,297.595228,50.695744,216.879975,92.003321,147.232333,447.136288,197.733468
2024-07-31,172.387380,301.133781,51.929090,218.921575,93.249551,143.574467,445.029425,198.101436
2024-08-01,171.675525,301.230690,52.322675,214.522941,93.490471,142.852517,424.574719,202.102019
2024-08-02,173.004445,305.457148,54.791940,222.783991,95.416854,140.272264,437.410995,203.564833
2024-08-05,172.625897,306.969464,54.067951,214.437258,95.956780,141.592000,411.076637,201.041657
2024-08-06,174.433143,312.998164,57.017008,217.222890,98.484813,143.397670,400.639382,203.219831
2024-08-07,173.583415,317.735980,57.871034,219.018436,99.469879,141.278802,401.383947,208.347454
2024-08-08,171.677049,311.700707,56.043363,207.987921,100.135160,145.553058,374.063634,212.421774
2024-08-09,170.808512,311.003839,55.465665,203.733180,101.783429,145.020828,362.800847,217.254988
2024-08-12,169.254579,312.580741,55.712262,209.084349,99.268539,142.156895,380.097701,215.962928
2024-08-13,169.819915,313.464446,56.005378,208.235813,99.053615,142.491119,375.182633,222.827200
2024-08-14,169.896556,310.597281,55.569444,211.228998,98.381143,143.079238,388.737550,223.084026
2024-08-15,169.845471,307.003203,54.348017,210.811500,96.443614,144.266809,395.590207,223.533867
2024-08-16,171.627978,311.502474,56.094730,214.920208,96.759030,144.142018,398.550256,228.313085
2024-08-19,171.397279,313.724770,56.652068,218.007022,97.157844,142.250746,405.976933,234.317996
2024-08-20,171.536687,316.237062,57.398661,216.371183,99.410578,142.706221,395.024924,238.447761
2024-08-21,170.827108,315.757876,57.536517,214.672330,98.937790,143.466412,388.070918,239.556937
2024-08-22,169.434150,316.052028,56.659792,213.011362,97.556598,143.456763,387.885781,239.945240
2024-08-23,168.321741,315.532933,55.902832,210.069308,96.609568,142.711939,382.360038,242.009887
2024-08-26,166.101880,316.548155,54.522259,204.761414,96.271429,141.572837,372.489023,247.915553
2024-08-27,166.271897,314.369331,52.992903,197.517277,96.894718,140.688376,356.707668,241.958648
2024-08-28,167.088876,316.178661,,196.673614,97.988453,142.277610,349.170981,237.912393
2024-08-29,168.014919,315.660590,53.592844,197.579911,97.920662,142.872439,353.089722,235.527435
2024-08-30,169.280243,315.605521,54.285515,200.411530,96.885796,146.416510,358.627933,233.004066
2024-09-02,168.786675,312.497311,52.918724,195.886994,96.112008,147.245172,351.430228,230.241234
2024-09-03,170.377636,310.640329,52.141242,194.947099,95.612595,147.742812,353.117547,227.504862
2024-09-04,171.462690,310.730731,52.500266,195.476442,95.467436,144.997780,352.667261,223.722472
2024-09-05,171.315996,,53.162455,196.598841,96.862373,144.132762,352.379506,220.771587
2024-09-06,171.453046,311.658115,52.389133,193.297835,96.052557,145.207961,345.704757,215.910382
2024-09-09,170.095044,310.580151,51.302431,191.873164,94.328423,146.400437,347.657910,215.859594
2024-09-10,168.767948,309.585049,50.685527,189.194196,94.266507,145.575018,342.157979,216.490951
2024-09-11,168.227376,308.995774,50.688168,187.964617,95.570567,145.878786,337.794493,218.412528
2024-09-12,167.752825,309.068272,50.668642,182.723470,98.508182,148.342823,319.691754,218.951837
2024-09-13,168.930691,311.433659,51.795649,183.629729,100.670767,149.268878,316.056802,221.672232
2024-09-16,167.668622,312.690836,50.720510,180.198315,101.529503,147.779294,310.780883,224.620879
2024-09-17,168.629042,312.585256,51.799571,180.619244,102.001351,150.026947,305.954700,227.629629
2024-09-18,167.474180,313.732500,51.324994,177.015146,102.375643,148.526179,296.711055,227.232477
2024-09-19,166.870692,307.797329,50.191381,177.471006,101.804176,148.242412,304.679523,220.136360
2024-09-20,166.559304,307.879051,49.488218,172.991480,99.113446,148.441867,293.738122,222.447065
2024-09-23,163.876837,311.095620,50.522674,176.798806,97.750063,144.243706,300.515306,226.617271
2024-09-23,163.876837,311.095620,50.522674,176.798806,97.750063,144.243706,300.515306,226.617271
2024-09-24,163.845774,310.238446,50.164300,174.031856,99.085807,144.545747,293.362383,226.273652
2024-09-25,162.535458,310.245561,50.517528,173.117082,99.870738,143.876102,288.395641,233.995215
2024-09-26,161.333618,316.329529,50.195923,172.830264,102.791083,141.509203,289.228815,236.484037
2024-09-27,159.816267,318.124647,49.740859,170.595909,101.769248,139.936858,284.420480,238.044217
2024-09-30,160.796476,321.590441,49.912076,173.414697,102.276388,138.618132,292.740295,245.318249
2024-10-01,161.516616,320.034322,50.506025,173.272618,104.016063,140.591248,288.967135,239.679389
2024-10-02,162.116824,316.328722,50.772363,170.132138,104.000322,144.164046,277.381953,242.035496
2024-10-03,160.634159,318.638720,50.664693,172.917682,104.463057,144.351567,286.941594,240.297671
2024-10-04,161.490704,316.484601,50.741998,172.095423,105.909236,144.751223,283.856167,237.821539
2024-10-07,159.661937,313.728754,49.219833,165.803729,103.812118,145.906398,271.701037,238.502040
2024-10-08,158.527122,313.058471,48.515565,164.687151,104.071105,144.978548,271.863443,235.209644
2024-10-09,159.556035,317.077724,49.079301,166.005797,104.516295,143.171760,273.099901,235.920978
2024-10-10,158.606272,319.381425,49.672740,168.416359,104.175433,142.360397,277.704557,232.902675
2024-10-11,158.992160,319.462560,49.340126,165.535822,104.919925,141.569523,270.206439,233.296880
2024-10-14,157.973707,318.738384,48.044152,165.627149,102.797444,141.780703,277.505609,231.879951
2024-10-15,157.592522,314.635642,47.861681,162.452223,103.345067,143.731703,268.149785,232.209918
2024-10-16,156.869092,313.918281,48.060804,164.292082,101.618696,142.967290,273.045628,238.127837
2024-10-17,155.897890,313.409179,47.213028,168.126804,100.258595,143.202127,290.608953,230.146970
2024-10-18,154.453121,314.229662,46.812730,168.759877,100.490878,143.649427,295.163289,236.060161
2024-10-21,154.086713,308.685448,45.741972,168.830641,99.490595,144.424302,302.046871,236.951262
2024-10-22,153.499467,312.375678,46.624701,170.037338,99.414008,141.614947,300.720793,245.921508
2024-10-23,156.580523,313.121984,46.536537,170.569330,98.434613,144.211803,303.119108,245.580037
2024-10-24,157.848263,312.857195,46.595398,170.222007,98.779415,145.088393,301.543601,251.977653
2024-10-25,156.353352,312.432150,45.768932,167.989652,97.716355,147.500403,298.934505,252.139395
2024-10-28,157.150065,313.083871,45.695282,168.076628,99.116898,146.214715,299.701429,254.700791
2024-10-29,157.260975,313.691198,45.610424,168.708810,98.273769,146.049644,302.455400,252.960666
2024-10-30,158.596050,311.594003,46.023967,171.275330,97.189573,148.808061,308.853187,251.374713
2024-10-31,158.741599,314.429963,47.642366,172.007264,97.641783,148.022996,301.274754,256.905484
2024-11-01,159.747385,318.265935,49.029783,177.454027,99.080028,148.423732,311.553170,254.747006
2024-11-04,160.199275,321.090251,49.428609,179.468731,99.995528,147.845285,316.056938,258.545456
2024-11-05,156.761526,324.508888,48.790666,177.440349,98.980850,145.670708,312.972727,260.453650
2024-11-06,155.785078,323.000496,47.277145,175.352072,97.341740,145.545958,315.191127,259.343294
2024-11-07,157.371079,323.027252,47.057204,174.345754,97.542567,148.925068,313.053567,255.536227
2024-11-08,157.049711,324.230425,46.997754,173.167002,97.262631,147.414410,309.292368,258.618463
2024-11-11,157.660862,324.757910,46.747943,174.703435,97.656669,145.382201,316.311659,257.692312
2024-11-12,158.884781,323.053390,45.654736,176.166314,97.983839,146.586971,328.919555,254.627850
2024-11-13,157.556406,318.272996,44.903583,176.461728,96.817441,151.712424,335.304031,247.256166
2024-11-14,159.475466,318.079012,46.565829,179.374214,96.235968,153.823253,334.357959,249.008866
2024-11-15,158.326056,316.040230,45.812622,178.319989,95.751828,155.172363,335.734953,244.122114
2024-11-18,157.124160,310.256524,44.896389,171.999956,94.967606,156.885947,318.931494,233.569064
2024-11-19,159.570589,310.626859,45.285424,172.589747,94.973450,157.536873,318.428798,232.448437
2024-11-20,159.888111,309.465900,44.237062,171.559920,93.065262,159.698550,321.876456,233.268809
2024-11-21,159.784057,309.012570,43.848937,171.588573,93.152691,158.921998,324.719626,229.717744
2024-11-22,161.675571,314.421527,45.734111,175.240555,95.083087,155.830872,324.996072,232.714551
2024-11-25,161.931365,316.608019,46.341456,180.114136,95.462161,155.278308,338.635545,227.818246
2024-11-26,162.538223,316.062093,45.827241,179.427570,96.100729,155.401433,339.731846,227.773280
2024-11-27,162.778403,315.346548,45.771355,179.678287,94.938318,157.193203,341.062907,222.296914
2024-11-28,163.450154,316.489720,45.810471,180.302903,94.300926,157.904657,343.109847,220.471507
2024-11-29,163.401028,314.462157,45.174215,179.431328,93.998692,162.102583,344.463145,224.612941
2024-12-02,162.785261,310.940423,44.463660,174.551096,94.210034,163.561617,331.341932,222.710988
2024-12-03,162.888164,306.472742,42.643976,171.286632,92.991206,169.413154,332.380520,224.808467
2024-12-04,164.905178,308.308745,43.166397,171.227934,92.692434,170.968358,328.294014,220.512381
2024-12-05,163.935476,311.078785,43.145085,172.991621,93.071914,170.434997,335.116181,226.546335
2024-12-06,165.450093,319.021072,44.740483,175.833370,95.152470,169.341960,334.133048,229.299345
2024-12-09,167.541204,316.365242,45.079142,173.018837,96.900470,170.856458,321.389643,225.854043
2024-12-10,169.003087,315.396458,44.941933,173.237155,96.597184,174.013603,323.130304,224.670334
2024-12-11,169.982353,316.728996,45.035526,175.357454,96.890748,173.567206,330.269773,225.750389
2024-12-12,169.559858,312.136392,44.185537,173.526137,96.190235,177.560125,329.518304,220.821809
2024-12-13,170.506500,312.692538,44.507444,176.958631,97.450209,174.797420,340.040590,223.313838
2024-12-16,172.495459,311.070898,44.877482,179.727544,98.270659,178.240574,347.774145,221.716431
2024-12-17,174.392522,308.177986,45.041220,181.403358,98.614879,182.725224,352.923810,218.935601
""")
Path("sample/prices.csv").write_text("""date,AAA,BBB,CCC,DDD,EEE,FFF
2024-01-02,101.20,55.10,78.40,142.70,23.90,310.50
2024-01-03,100.31,53.80,77.12,142.70,23.89,311.12
2024-01-04,99.72,53.00,77.61,144.92,24.07,310.48
2024-01-05,98.67,53.61,78.17,141.03,24.29,305.02
2024-01-08,98.00,52.98,76.67,141.47,23.91,303.80
2024-01-09,98.48,53.76,74.94,141.65,23.32,302.26
2024-01-10,97.08,53.14,74.33,139.93,22.71,300.77
2024-01-11,95.92,51.72,73.62,137.06,22.67,301.25
2024-01-12,93.80,50.95,71.40,135.38,22.27,295.21
2024-01-15,93.58,50.68,72.49,133.21,22.62,291.70
2024-01-16,93.25,50.01,73.98,133.70,23.80,294.56
2024-01-17,94.18,50.96,73.90,134.48,24.58,293.12
2024-01-18,94.41,51.05,74.83,134.04,24.67,293.80
2024-01-19,94.55,50.57,75.78,132.02,24.04,290.20
2024-01-22,95.62,50.81,74.75,131.24,23.88,286.21
2024-01-23,94.28,50.61,72.83,129.75,23.56,283.74
2024-01-24,94.25,51.46,72.68,127.85,22.92,281.63
2024-01-25,93.90,51.48,72.03,126.89,22.51,279.68
2024-01-26,94.10,51.32,72.10,129.98,22.81,278.34
2024-01-29,96.00,52.42,71.77,133.13,22.95,277.87
2024-01-30,95.34,51.47,71.19,134.29,22.58,280.21
2024-01-31,97.42,51.61,71.88,135.56,22.08,278.61
2024-02-01,97.43,51.90,71.37,135.73,22.36,278.96
2024-02-02,98.42,52.65,72.60,139.56,23.13,281.68
2024-02-05,99.25,52.41,72.11,141.67,23.21,286.38
2024-02-06,99.29,51.48,74.17,143.37,22.97,292.58
2024-02-07,99.36,50.34,72.89,142.62,22.99,290.08
2024-02-08,99.10,50.06,72.19,140.90,22.81,291.44
2024-02-09,99.84,49.79,70.60,141.88,22.07,289.96
2024-02-12,97.98,50.24,71.87,138.56,22.93,291.68
2024-02-13,95.39,48.79,70.14,137.43,22.42,291.21
2024-02-14,95.36,49.93,73.74,136.33,23.05,298.23
2024-02-15,94.33,49.13,73.14,136.23,23.27,299.56
2024-02-16,93.43,49.05,73.32,138.50,23.53,298.37
2024-02-19,92.36,49.48,72.74,134.18,23.26,295.16
2024-02-20,93.64,49.42,73.78,136.55,23.07,296.76
2024-02-21,95.57,51.37,76.50,138.74,24.53,297.36
2024-02-22,95.49,50.79,76.12,138.55,25.51,296.62
2024-02-23,96.91,52.28,77.03,141.81,25.14,298.62
2024-02-26,97.88,51.15,74.82,144.64,24.20,297.35
2024-02-27,97.50,50.64,75.45,145.02,24.12,296.82
2024-02-28,97.47,51.48,76.49,146.09,24.60,294.43
2024-02-29,96.71,51.57,76.61,143.83,23.95,293.71
2024-03-01,96.37,51.20,75.87,144.06,23.88,296.21
2024-03-04,97.42,52.54,78.03,144.37,24.55,297.58
2024-03-05,97.11,52.76,79.82,142.60,24.15,300.41
2024-03-06,97.16,53.26,78.23,145.80,24.47,301.20
2024-03-07,96.80,53.18,77.89,145.02,23.91,304.06
2024-03-08,96.78,53.22,78.06,140.98,23.98,300.46
2024-03-11,96.50,53.14,76.60,142.27,23.49,298.75
2024-03-12,95.59,52.83,77.34,140.38,22.95,301.15
2024-03-13,94.47,53.11,77.30,138.23,22.83,301.07
2024-03-14,95.45,53.44,79.03,142.55,23.22,301.67
2024-03-15,94.35,52.01,77.03,140.40,21.97,302.04
2024-03-18,91.62,49.92,78.14,139.44,22.13,305.18
2024-03-19,93.51,51.12,79.35,139.13,22.37,302.41
2024-03-20,92.79,50.34,78.66,137.95,22.42,298.61
2024-03-21,92.64,51.47,79.48,139.82,23.49,302.80
2024-03-22,93.45,50.95,78.86,139.97,22.46,301.86
2024-03-25,92.16,49.32,77.87,138.71,21.67,299.77
""")
Path("sample/expected/summary.csv").write_text("""asset,annual_vol,worst_day,max_drawdown
AAA,0.177722,-0.029362,0.094664
BBB,0.263690,-0.041014,0.114519
CCC,0.272770,-0.030619,0.105357
DDD,0.234827,-0.031688,0.124413
EEE,0.385845,-0.055336,0.150529
FFF,0.143058,-0.020254,0.106872
""")
Path("sample/expected/corr.csv").write_text("""1.000000,0.603091,0.299041,0.570979,0.180018,0.056452
0.603091,1.000000,0.469628,0.216222,0.442119,0.048120
0.299041,0.469628,1.000000,0.121569,0.566649,0.459137
0.570979,0.216222,0.121569,1.000000,0.250655,0.333888
0.180018,0.442119,0.566649,0.250655,1.000000,0.259759
0.056452,0.048120,0.459137,0.333888,0.259759,1.000000
""")
print("wrote: \n prices.csv \n sample/prices.csv \n sample/expected/summary.csv \n sample/expected/corr.csv")

wrote: 
 prices.csv 
 sample/prices.csv 
 sample/expected/summary.csv 
 sample/expected/corr.csv


### Data format

`prices.csv`: a header `date,AAA,BBB,...`, then one row per day with an ISO date (`2024-01-02`) and one price per asset. A clean file has dates in ascending order and every price positive. Lines may end with a line feed or with a carriage return and a line feed (Windows). Both sample and data have exactly the same format.

### Definitions

N is the number of log returns. Day 0 is the first row of prices, so the log returns are for days t = 1, ..., N.

- **Log return** of an asset on day t: `ln(price(t) / price(t-1))`.
- **annual\_vol**: sample standard deviation (divide by N-1) of the log returns, times the square root of 252.
- **worst\_day**: the smallest log return.
- **max\_drawdown**: for each asset on its own, let `peak(t)` be the highest of `price(0), price(1), ..., price(t)`. Then max\_drawdown is the largest value, over all days t, of `(peak(t) - price(t)) / peak(t)`.
- **Covariance and correlation**: for two assets, say A and B, `cov(A, B)` is the sum over the N days of `(A(t) - mean_A) * (B(t) - mean_B)`, divided by N-1, where `A(t)` is the log return of A on day t and `mean_A` is the mean of its N log returns. `corr(A, B) = cov(A, B) / sqrt(cov(A, A) * cov(B, B))`.


# Code purpose

The following code will determine the correlation matrix from a portfolio of assets

In [ ]:
%%writefile src/PriceSeries.hpp
#pragma once
#include <string>
#include <vector>

// PriceSeries class will handle individual assets
// Asset name, dates, and prices are invariant and are not able to be changed in a function that calls the class
class PriceSeries {
  private:
    std::string name;
    std::vector<std::string> dates;
    std::vector<double> prices;
  public:
    PriceSeries(
    const std::string& name,
    const std::vector<std::string>& dates,
    const std::vector<double>& prices
    );
    const std::string& get_name() const;
    const std::vector<std::string>& get_dates() const;
    const std::vector<double>& get_prices() const;
};

Overwriting src/PriceSeries.hpp


In [ ]:
%%writefile src/PriceSeries.cpp
#include "PriceSeries.hpp"
#include <stdexcept>
#include <cmath>

PriceSeries::PriceSeries(const std::string& name,
                         const std::vector<std::string>& dates,
                         const std::vector<double>& prices)
                         : name(name), dates(dates), prices(prices)
{
    // Quick invariant checks to make sure a Price Series is of the correct form
    // if not an error message is thrown and the rest of the code will not run
    if (prices.empty()) {
      throw std::invalid_argument("Prices cannot be empty");
    }

    if (prices.size() != dates.size()) {
      throw std::invalid_argument("There have to be as many prices as dates");
    }

    for (double price : prices) {
      if (price <= 0) {
        throw std::invalid_argument("Prices have to be strictly greater than zero");
      }
      if (std::isfinite(price) == false) {
        throw std::invalid_argument("Prices must be finite");
      }
    }
    for (std::size_t c = 0; c < dates.size(); c++) {
      for (std::size_t k = c + 1; k < dates.size(); k++) {
        if (dates[c] == dates[k]) {
          throw std::invalid_argument("There cannot be duplicate days");
        }
      }
    }

}
// initialize getters for PriceSeries object
const std::string& PriceSeries::get_name() const {
    return name;
}

const std::vector<std::string>& PriceSeries::get_dates() const {
    return dates;
}

const std::vector<double>& PriceSeries::get_prices() const {
    return prices;
}

Writing src/PriceSeries.cpp


In [ ]:
%%writefile src/CollectionSeries.hpp
#pragma once
#include <string>
#include <vector>
#include "PriceSeries.hpp"

// initialize a class that hold multiple PriceSeries objects, like a portfolio of assets
// since the PriceSeries object holds all its own invariants only series needs to be invariant in this class
class CollectionSeries {
  private:
    std::vector<PriceSeries> series;

  public:
    CollectionSeries(
      const std::vector<PriceSeries>& series
    );
    const std::vector<PriceSeries>& get_series() const;
};

Overwriting src/CollectionSeries.hpp


In [ ]:
%%writefile src/CollectionSeries.cpp
#include "CollectionSeries.hpp"
#include <stdexcept>
#include <cmath>

CollectionSeries::CollectionSeries(const std::vector<PriceSeries>& series): series(series)

{
  // again just a couple quick invariant checks to ensure the PriceSeries objects we recieve are of the form
  // required by the rest of the code
  if (series.empty()) {
    throw std::invalid_argument("Collection cannot be empty");
  }
  // checks to make sure every asset in the collection has a unique name
  for (std::size_t i = 0; i < series.size(); ++i) {
    for (std::size_t j = i + 1; j < series.size(); ++j )
      if (series[i].get_name() == series[j].get_name()) {
        throw std::invalid_argument("Cannot have duplicate asset sets");
      }
  }
  // this was added without realizing the specifications in Task 3 so this will throw an error in case
  // asset dates do not match up
  for (std::size_t i = 1; i < series.size(); ++i) {
    if (series[0].get_dates() != series[i].get_dates()) {
      throw std::invalid_argument("Asset dates do not match");
    }
  }
}
// getter that returns the individual PriceSeries (assets)
const std::vector<PriceSeries>& CollectionSeries::get_series() const {
    return series;
}

Writing src/CollectionSeries.cpp


In [ ]:
%%writefile src/CsvReader.hpp
#pragma once
#include <string>
#include "CollectionSeries.hpp"

CollectionSeries load_prices(const std::string& filename);

Overwriting src/CsvReader.hpp


In [ ]:
%%writefile src/CsvReader.cpp

#include "CsvReader.hpp"
#include <fstream>
#include <sstream>
#include <stdexcept>
#include <vector>
#include <cmath>
#include <iostream>

// trim functionality as given in the companion document
std::string trim(const std::string& s) {
    std::size_t first = s.find_first_not_of(" \t\r");
    if ( first == std::string::npos) return "" ;
    std::size_t last = s.find_last_not_of( " \t\r");
    return s.substr(first, last - first + 1);
}

CollectionSeries load_prices(const std::string& filename) {

    // Open the CSV file
    std::ifstream file(filename);

    if (file.is_open() == false) {
        throw std::runtime_error("Could not open file");
    }

    // Read the header
    std::string line;
    std::getline(file, line);

    std::stringstream header_stream(line);
    std::string field;

    std::vector<std::string> asset_names;

    // Since the header of the PriceSeries csv files is of the form (date, AAA, BBB, ...) we drop
    // the first field as it holds no value
    std::getline(header_stream, field, ',');

    // while loop that loops through the resulting header which will not only contain the asset names
    // and appends each name to the asset_names vector
    while (std::getline(header_stream, field, ',')) {
      asset_names.push_back(field);
    }

    // Initialize vectors for the dates and asset prices
    std::vector<std::string> dates;

    // set the size of asset_prices to be a vector of vectors with each interior vector having the same size as
    // asset_names. This is because asset_prices will contain all the assets prices.
    std::vector<std::vector<double>> asset_prices(asset_names.size());

    // With the header out of the way the remaining rows of the csv file are of the form
    // XXXX-YY-ZZ, price1, price2, ...
    // Task 3 modification: line_number will track which line were on so we can print which line was dropped and why

    std::size_t line_number = 1;
    while (std::getline(file, line)) {
        // Task 3 modification: Increase line number by 1 immedietly as line numbers dont start at 0
        ++line_number;
        // Task 3 modofication: If a row is invalid then valid_row = false and the logic for adding said row to the
        // date and asset_price vector is skipped
        bool valid_row = true;
        // Task 3 modification: create a string object which will hold reason a row is dropped
        std::string reason;

        // Takes entire row as a string and then splits it into a vector of strings where each entry is an
        // individual row variable
        std::stringstream ss(line);
        std::string cell;
        std::vector<std::string> cells;

        // only push the trimmed row elements into cells
        while (std::getline(ss, cell, ',')) {
            cells.push_back(trim(cell));
        }


        // Task 3 modification: START ################################################################################
        // Recall that "date" was dropped from the header so to check the size of each row to the size of the header
        // we must add 1
        if (cells.size() != (asset_names.size() + 1)) {
          valid_row = false;
          reason = "incorrect number of columns";
        }
        // Here we check to make sure the row fields are empty if its not empty we move on and check whether the
        // price fields are real
        for (std::size_t c = 0; c < cells.size(); c++) {
          if (cells[c].empty()) {
            valid_row = false;
            if (c == 0) {
              reason = "empty date field";
            }
            else {
              // again since the date was dropped from the header we have to backstep the asset_names vector by 1
              reason = "empty cell in column " + asset_names[c - 1];
            }
            break;
          }

          // Now since were still in the loop and the date field is good we check to make sure the price values
          // are real allowed values
          if (c != 0 && valid_row) {
            try{
              // the following code was formulated using the companion document
              std::size_t used = 0;
              double x = std::stod(cells[c], &used);
              valid_row = (used == cells[c].size()) && std::isfinite(x) && (x > 0.0);
              if (valid_row == false) {
                reason = "invalid price in column " + asset_names[c - 1];
                break;
              }
            }
            catch (const std::exception&) {
              valid_row = false;
              reason = "invalid price in column " + asset_names[c - 1];
              break;
            }
          }
        }

        // If all the checks passed then the valid_row bool will still remain true and the remainder of the code
        // adding it to the date and asset_price vectors runs. If not, then the reason is printed and we move on to the
        // next line using the continue line
        if (valid_row == false) {
          std::cout << "line " << line_number << ": " << reason << ", row dropped" << std::endl;
          continue;
        }
        // Task 3 modifcation: END ###################################################################################

        std::stringstream row_stream(line);

        // First field is the date so we add it to the date vector
        std::getline(row_stream, field, ',');
        dates.push_back(field);

        // Remaining fields are asset prices so we loop through them and add each price to the respective asset
        // name column
        // Also note that here we no longer have to adjust the bound of the loop as the price vector rows only contain as
        // many elements as the asset names
        for (std::size_t j = 0; j < asset_names.size(); j++) {

          std::getline(row_stream, field, ',');

          // now that we know that each price is real
          double price = std::stod(field);
          // add price to jth column
          asset_prices[j].push_back(price);
          }
    }
    // Task 3 modification: Now that the date and asset vectors contain all their elements we implement
    // the final rule; if theres fewer than 3 assets we dont even bother with the rest of the analysis
    if (dates.size() < 3) {
      throw std::runtime_error("fewer than 3 valid rows remain after cleaning");
    }

    // Here we construct the sort algorithm which has to move around the asset_price vectors as well
    for (std::size_t i = 1; i < dates.size(); ++i) {

      // Save the current date
      std::string current_date = dates[i];

      // Save all asset prices corresponding to this date
      std::vector<double> current_prices(asset_names.size());

      for (std::size_t a = 0; a < asset_names.size(); ++a) {
        current_prices[a] = asset_prices[a][i];
      }
      std::size_t j = i;

      // Shift dates and their corresponding prices to the right
      while (j > 0 && dates[j - 1] > current_date) {
        dates[j] = dates[j - 1];
        for (std::size_t a = 0; a < asset_names.size(); ++a) {
          asset_prices[a][j] = asset_prices[a][j - 1];
        }
        --j;
      }

      // Insert the current date and its prices into the gap
      dates[j] = current_date;

      for (std::size_t a = 0; a < asset_names.size(); ++a) {
        asset_prices[a][j] = current_prices[a];
      }
    }

    // With the dates sorted we do only final check; if a date contains a duplicate date then only the first mention
    // of said date is kept
    // Since the date vector along with the asset_prices are already sorted we just drop a row if the row before it
    // has the same date
    for (std::size_t i = 1; i < dates.size();) {
      if (dates[i] == dates[i - 1]) {
        std::cout << "Duplicate date " << dates[i] << ", row dropped\n";
          // Remove the duplicate date
          dates.erase(dates.begin() + i);
          // Remove the corresponding asset prices
          for (std::size_t j = 0; j < asset_names.size(); ++j) {
              asset_prices[j].erase(asset_prices[j].begin() + i);
          }
      }
      else {
        ++i;
      }
    }

    // Since we now have all the dates and prices we can construct our PriceSeries object
    // Initialize the PriceSeries object
    std::vector<PriceSeries> series;

    // Loop through the asset_names and add to the back of it the dates and prices
    for (std::size_t j = 0; j < asset_names.size(); j++) {
        series.emplace_back(asset_names[j], dates, asset_prices[j]);
    }

    // With all this PriceSeries object now we can finally construct the collection object
    return CollectionSeries(series);

}

Writing src/CsvReader.cpp


In [ ]:
%%writefile src/Calculations.hpp
#pragma once
#include "CollectionSeries.hpp"
#include <Eigen/Dense>

// Calculate daily log returns
Eigen::MatrixXd log_returns(const CollectionSeries& collection);

// Calculate sample covariance matrix
Eigen::MatrixXd calculate_covariance(const Eigen::MatrixXd& returns);

// Calculate correlation matrix
Eigen::MatrixXd calculate_correlation(const Eigen::MatrixXd& covariance);

// Calculate the max drawdown
double max_drawdown(const std::vector<double>& prices);

// Annualization days
double annualisation_days();

Overwriting src/Calculations.hpp


In [ ]:
%%writefile src/Calculations.cpp

#include "Calculations.hpp"
#include <cmath>
#include <stdexcept>

Eigen::MatrixXd log_returns(const CollectionSeries& collection) {
    // auto lets the compiler decide the type of the collection.get_series()
    // grab the series
    const auto& series = collection.get_series();

    // series.size() is the amount of assets in the collection
    const std::size_t assets = series.size();
    // to obtain the amount of observations we just look at one of the asset columns and subtract 1 from its size
    // since the first element is just the asset name
    const std::size_t observations = series[0].get_prices().size() - 1;

    // initialize a dynamic Eigen matrix object with size observations x rows
    Eigen::MatrixXd returns(observations, assets);

    // loop over the different assets
    for (std::size_t j = 0; j < assets; ++j) {
        // get the prices for the current asset
        const auto& prices = series[j].get_prices();
        // compute log returns between asset prices
        for (std::size_t i = 0; i < observations; ++i) {
            returns(i, j) = std::log(prices[i + 1] / prices[i]);
        }
    }
    return returns;
}

Eigen::MatrixXd calculate_covariance(const Eigen::MatrixXd& returns) {
    // Calculate the mean of each column
    Eigen::RowVectorXd means = returns.colwise().mean();
    // Subtract the means from each observation as in the previous lab
    Eigen::MatrixXd centered = returns.rowwise() - means;
    // Calculate covariance matrix of the returns
    Eigen::MatrixXd covariance = (centered.transpose() * centered) / (returns.rows() - 1);
    return covariance;
}

Eigen::MatrixXd calculate_correlation(const Eigen::MatrixXd& covariance) {
    // To apply the size of our covariance matrix as an index on the loop below we
    // must use Eigen::Index
    const Eigen::Index n = covariance.rows();

    // Initialize the correlation matrix with size equal to the amount of assets
    Eigen::MatrixXd correlation(n, n);

    for (Eigen::Index i = 0; i < n; ++i) {
        for (Eigen::Index j = 0; j < n; ++j) {

            double denominator = std::sqrt(covariance(i, i) * covariance(j, j));

            if (denominator == 0.0) {
              correlation(i, j) = 0.0;
            }
            else {
                correlation(i, j) = covariance(i, j) / denominator;
            }
        }
    }
    return correlation;
}

double max_drawdown(const std::vector<double>& prices) {
    double peak = prices[0];
    double worst = 0.0;
    for (double price : prices) {
        if (price > peak) {
            peak = price;
        }
        double drawdown = (peak - price) / peak;
        if (drawdown > worst) {
            worst = drawdown;
        }
    }
    return worst;
}

double annualisation_days() {
    return 252.0;
}

Writing src/Calculations.cpp


In [ ]:
%%writefile src/main.cpp

#include "CsvReader.hpp"
#include "Calculations.hpp"

#include <cmath>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <algorithm>
#include <stdexcept>

int main() {
    try {
        // Load price data
        double days = annualisation_days();
        CollectionSeries collection = load_prices("prices.csv");
        const auto& series = collection.get_series();
        std::cout << "Loaded " << series[0].get_prices().size() << " rows, " << series.size() << " assets" << std::endl;
        std::cout << "Annualising over " << days << " trading days" << std::endl;

        // Calculate returns, covariance and correlation
        Eigen::MatrixXd returns = log_returns(collection);
        Eigen::MatrixXd cov = calculate_covariance(returns);
        Eigen::MatrixXd corr = calculate_correlation(cov);

        // Write summary.csv
        std::ofstream summary("summary.csv");
        summary << std::fixed << std::setprecision(6);
        summary << "asset,annual_vol,worst_day,max_drawdown\n";
        for (Eigen::Index c = 0; c < returns.cols(); ++c) {
            const auto& prices = series[c].get_prices();
            double vol = std::sqrt(cov(c, c) * days);
            double worst = returns.col(c).minCoeff();
            double mdd = max_drawdown(prices);
            summary << series[c].get_name() << "," << vol << "," << worst << "," << mdd << "\n";
            }
        summary.close();

        // Write corr.csv
        std::ofstream corr_out("corr.csv");
        corr_out << std::fixed << std::setprecision(6);
        for (Eigen::Index i = 0; i < corr.rows(); ++i) {
            for (Eigen::Index j = 0; j < corr.cols(); ++j) {
                corr_out << corr(i, j);
                if (j + 1 < corr.cols()) {
                    corr_out << ",";
                }
                else {
                    corr_out << "\n";
                }
            }
        }
        corr_out.close();
    }
    catch (const std::exception& e) {
        std::cout << "error: " << e.what() << std::endl;
        return 1;
    }
    std::cout << "Done. \n" << std::endl;
    return 0;
}

Writing src/main.cpp


In [ ]:
!g++ -std=c++17 -Wall -Wextra -Werror -isystem eigen-3.4.0 src/*.cpp -o cw1

In [ ]:
!rm -rf sample_run && mkdir sample_run && cp sample/prices.csv sample_run/
!cd sample_run && ../cw1
!python3 check_outputs.py sample_run sample/expected

Loaded 60 rows, 6 assets
Annualising over 252 trading days
Done. 

PASS  summary.csv  largest difference 0.0e+00
PASS  corr.csv     largest difference 0.0e+00


# Design notes

### Class: _PriceSeries_
**Purpose:** The purpose of this class is to represent an assets price data and guarantee that its basic data invariants are satisfied.

 **Invariants:** The names, dates, and prices of an asset are invariant and thus cannot be changed outside the class. This is necessary as the data is based on true observation and so a change is not permissable

 **Invariant Checks:** When the PriceSeries class is initialized with a data set the following checks are enfored so the rest of the analysis based on the data will work as intended.


1.   Prices cannot be empty
2.   There has to be as many prices as dates
3.   Prices must be strictly greater than zero
4.   Prices may not be infinite
5.   Each day is unique i.e there cannot be two rows with the   same date

**Getters:** In order to extract information about the PriceSeries object I create the following getters

1. ```get_name()``` (returns the asset name)
2. ```get_dates()``` (returns the date column)
3. ```get_prices()``` (returns the prices associated with the asset)
****
### Class: _CollectionSeries_
**Purpose:** This class represents a collection of individual assets allowing us to perform portfolio calculations like correlation and covariance of a portfolio of assets.

 **Invariants:** This class only holds the individual series as invariants. This means that a given series cannot be modified in any way, although we do not require additional invariants like names, dates, or prices as that is already included in the PriceSeries class.

 **Invariant Checks:** When this class is initialized with a colleciton of assets we check the following invariants about each series.

1.   The collection itself cannot be empty; there must be at least one asset
2.   The collection cannot contain more than one of an individual asset
3.   Although this has already been taken care of in Task 3, we have an invariant check that each asset within the collection must match the dates of the other assets in the collection

**Getters:** The only getter we require here is one that just returns the series, this is because any further information about the series can be obtained through the getters in PriceSeries.

1. ```get_series()``` (returns the series within the collection)
****
### Component: _CsvReader_
 **Purpose:** This components purpose is to read and clean an input file, construct a PriceSeries object for each asset in the input file, and then package them together and put them into a Collection Series object. Separating the reading and preprocessing functionality from the data classes means that the classes themselves do not need to know anything about CSV files or file handling. Once the data has been processed it is stored in a CollectionSeries object.

 **Trimming:** Within the CsvReader.cpp source file we also include a trimming function that removes whitespace etc. from the data within the csv files, the code for this was provided by the companion document.
****
### Component: _Calculations_
**Purpose:** This component contains the mathematical and financial functions used throughout the program. These functions operate on the validated data structures and perform calculations such as log returns, covariance, correlation and maximum drawdown.
****
### Component: _Main_
**Purpose:** This component executed the program by calling CsvReader to read a file, passes the resulting individual data to PriceSeries and then bundles those assets into a collection. Once that is done it passes the collection to the various calculation functions for analysis and writes the financial statistics into the two output csv files summary.csv and corr.csv.

### Doc tests for data cleanup logic


In [ ]:
%%writefile tests/test_engine.cpp
#define DOCTEST_CONFIG_IMPLEMENT_WITH_MAIN
#include "doctest.h"
#include "src/CsvReader.hpp"
#include <stdexcept>
#include <fstream>
#include <cstdio>
#include <string>
#include <fstream>
#include <cstdio>
#include <string>
#include <cmath>

// Helper function to create small CSV files for testing
void write_csv(const std::string& filename, const std::string& contents) {
    std::ofstream file(filename);
    file << contents;
}

TEST_CASE("1. Bad rows are removed") {
    write_csv("test.csv",
        "Date,AAPL,MSFT\n"
        "2025-01-01,100,200\n"
        "2025-01-02,,210\n"
        "2025-01-03,N/A,220\n"
        "2025-01-04,0,230\n"
        "2025-01-05,105,240,extra\n"
        "2025-01-06,110,250\n"
        "2025-01-07,115,260\n"
    );
    CollectionSeries collection = load_prices("test.csv");
    const auto& series = collection.get_series();
    CHECK(series.size() == 2);
    CHECK(series[0].get_dates().size() == 3);
    CHECK(series[0].get_prices()[0] == doctest::Approx(100));
    CHECK(series[0].get_prices()[1] == doctest::Approx(110));
    CHECK(series[0].get_prices()[2] == doctest::Approx(115));
    std::remove("test.csv");
}


TEST_CASE("2. Rows are sorted by date") {
    write_csv("test.csv",
        "Date,AAPL,MSFT\n"
        "2025-01-03,300,400\n"
        "2025-01-01,100,200\n"
        "2025-01-04,400,500\n"
        "2025-01-02,200,300\n"
    );
    CollectionSeries collection = load_prices("test.csv");
    const auto& series = collection.get_series();
    const auto& dates = series[0].get_dates();
    CHECK(dates[0] == "2025-01-01");
    CHECK(dates[1] == "2025-01-02");
    CHECK(dates[2] == "2025-01-03");
    CHECK(dates[3] == "2025-01-04");
    std::remove("test.csv");
}


TEST_CASE("3. Duplicate dates keep the first row") {
    write_csv("test.csv",
        "Date,AAPL,MSFT\n"
        "2025-01-01,100,200\n"
        "2025-01-02,110,210\n"
        "2025-01-02,999,999\n"
        "2025-01-03,120,220\n"
    );
    CollectionSeries collection = load_prices("test.csv");
    const auto& series = collection.get_series();
    CHECK(series[0].get_dates().size() == 3);
    CHECK(series[0].get_prices()[1] == doctest::Approx(110));
    CHECK(series[1].get_prices()[1] == doctest::Approx(210));
    std::remove("test.csv");
}


TEST_CASE("4. Blank lines are ignored") {
    write_csv("test.csv",
        "Date,AAPL,MSFT\n"
        "2025-01-01,100,200\n"
        "\n"
        "     \n"
        "2025-01-02,110,210\n"
        "2025-01-03,120,220\n"
    );
    CollectionSeries collection = load_prices("test.csv");
    const auto& series = collection.get_series();
    CHECK(series[0].get_dates().size() == 3);
    CHECK(series[0].get_prices()[0] == doctest::Approx(100));
    CHECK(series[0].get_prices()[1] == doctest::Approx(110));
    CHECK(series[0].get_prices()[2] == doctest::Approx(120));
    std::remove("test.csv");
}


TEST_CASE("5. Fewer than three valid rows produces an error") {
    write_csv("test.csv",
        "Date,AAPL,MSFT\n"
        "2025-01-01,100,200\n"
        "2025-01-02,,210\n"
        "2025-01-03,120,220\n"
    );
    CHECK_THROWS(load_prices("test.csv"));
    std::remove("test.csv");
}



Overwriting tests/test_engine.cpp


### Run tests


In [ ]:
!g++ -std=c++17 -Wall -Wextra -isystem eigen-3.4.0 -I. -Isrc tests/*.cpp $(ls src/*.cpp | grep -v main.cpp) -o tests_run && ./tests_run

[doctest] doctest version is "2.5.3"
[doctest] run with "--help" for options
line 3: empty cell in column AAPL, row dropped
line 4: invalid price in column AAPL, row dropped
line 5: invalid price in column AAPL, row dropped
line 6: incorrect number of columns, row dropped
Duplicate date 2025-01-02, row dropped
line 3: incorrect number of columns, row dropped
line 4: empty date field, row dropped
line 3: empty cell in column AAPL, row dropped
[doctest] test cases:  5 |  5 passed | 0 failed | 0 skipped
[doctest] assertions: 17 | 17 passed | 0 failed |
[doctest] Status: SUCCESS!


### Run code

In [ ]:
!g++ -std=c++17 -Wall -Wextra -Werror -isystem eigen-3.4.0 src/*.cpp -o cw1
!./cw1
!head -n 8 corr.csv

line 13: invalid price in column CCC, row dropped
line 50: empty cell in column EEE, row dropped
line 173: empty cell in column CCC, row dropped
line 179: empty cell in column BBB, row dropped
Duplicate date 2024-09-23, row dropped
Loaded 247 rows, 8 assets
Annualising over 252 trading days
Done. 

1.000000,0.138027,0.584628,0.504534,0.311591,0.082521,0.255644,-0.047724
0.138027,1.000000,0.621388,0.440832,0.405873,-0.632647,0.143540,0.385419
0.584628,0.621388,1.000000,0.683412,0.495147,-0.334471,0.196050,0.295513
0.504534,0.440832,0.683412,1.000000,0.210778,-0.320293,0.849849,0.090785
0.311591,0.405873,0.495147,0.210778,1.000000,-0.252899,-0.074337,0.173499
0.082521,-0.632647,-0.334471,-0.320293,-0.252899,1.000000,-0.188761,-0.294594
0.255644,0.143540,0.196050,0.849849,-0.074337,-0.188761,1.000000,-0.091393
-0.047724,0.385419,0.295513,0.090785,0.173499,-0.294594,-0.091393,1.000000


# Preprocessing of `prices.csv`

**Modifications:** Within the CsvReader function I added all the logic to drop rows based on the 4 rules above as well as the 5th rule where if the resulting, cleaned, data has less than 3 observations then nothing is returned.

**Results:** After applying the dropping logic and running the program on the `prices.csv` results in the following output
```
line 13: invalid price in column CCC, row dropped
line 50: empty cell in column EEE, row dropped
line 173: empty cell in column CCC, row dropped
line 179: empty cell in column BBB, row dropped
Duplicate date 2024-09-23, row dropped
```
**Confirmation:** Looking at the 13th of the `prices.csv` data
```
2024-01-17,155.990031,248.826870,-43.155231,156.372375,80.902403,144.463283,274.844728,211.279500
```
we see that the 3rd price entry is negative so that row should, and was, dropped.


---
---
# Checking for singular correlation matrices using precision of calculations


In [ ]:
%%writefile src/correlation_check.cpp
#include "CsvReader.hpp"
#include <iostream>
#include <fstream>
#include <sstream>
#include <Eigen/Eigenvalues>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>

std::vector<std::string> find_tied_assets(const Eigen::MatrixXd& corr, const std::vector<std::string>& names) {
  Eigen::SelfAdjointEigenSolver<Eigen::MatrixXd> solver(corr);

  std::vector<std::string> tied;

  if (solver.info() != Eigen::Success) {
    return tied;
  }

  Eigen::VectorXd eigenvalues = solver.eigenvalues();
  Eigen::MatrixXd eigenvectors = solver.eigenvectors();

  for (int j = 0; j < eigenvalues.size(); ++j) {

    double ratio = std::abs(eigenvalues[j] / eigenvalues[eigenvalues.size() - 1]);
    if (ratio < 1e-6) {
      for (int i = 0; i < eigenvectors.rows(); ++i) {
        if (std::abs(eigenvectors(i, j)) > 1e-6) {
          tied.push_back(names[i]);
        }
      }
    }
  }

  return tied;
}

int main() {
  CollectionSeries collection = load_prices("prices.csv");
  const auto& series = collection.get_series();

  std::vector<std::string> asset_names;

  for (std::size_t i = 0; i < series.size(); i++) {
    asset_names.push_back(series[i].get_name());
  }

  std::ifstream file("corr.csv");
  std::string line;
  std::vector<std::vector<double>> data;
  if (!file.is_open()) {
    std::cerr << "Could not open correlation file" << std::endl;
    return 1;
  }
  while (std::getline(file, line)) {
    std::stringstream lineStream(line);
    std::string cell;
    std::vector<double> row;
      while (std::getline(lineStream, cell, ',')) {
        if (!cell.empty()) {
          row.push_back(std::stod(cell)); // Breyta streng í double
        }
      }
      if (!row.empty()) {
        data.push_back(row);
      }
  }
  file.close();
  int rows = data.size();
  int cols = data[0].size();
  Eigen::MatrixXd corr(rows, cols);

  for (int i = 0; i < rows; i++) {
    for (int j = 0; j < cols; j++) {
      corr(i,j) = data[i][j];
    }
  }

  std::vector<std::string> tied = find_tied_assets(corr, asset_names);

  for (std::size_t i = 0; i < tied.size(); i++) {
    std::cout << tied[i] << std::endl;
  }
  return 0;
}

Writing src/correlation_check.cpp


In [ ]:
%%writefile tests/test_engine.cpp
#define DOCTEST_CONFIG_IMPLEMENT_WITH_MAIN
#include "doctest.h"
#include <Eigen/Eigenvalues>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>

std::vector<std::string> find_tied_assets0(const Eigen::MatrixXd& corr, const std::vector<std::string>& names) {
  Eigen::SelfAdjointEigenSolver<Eigen::MatrixXd> solver(corr);

  std::vector<std::string> tied;

  if (solver.info() != Eigen::Success) {
    return tied;
  }

  Eigen::VectorXd eigenvalues = solver.eigenvalues();
  Eigen::MatrixXd eigenvectors = solver.eigenvectors();

  for (int j = 0; j < eigenvalues.size(); ++j) {

    double ratio = std::abs(eigenvalues[j] / eigenvalues[eigenvalues.size() - 1]);
    if (ratio < 1e-6) {
      for (int i = 0; i < eigenvectors.rows(); ++i) {
        if (std::abs(eigenvectors(i, j)) > 1e-6) {
          tied.push_back(names[i]);
        }
      }
    }
  }

  return tied;
}

TEST_CASE("1. Degenerate correlation matrix") {
    Eigen::MatrixXd corr(3, 3);
    corr << 1, 1, 0,
            1, 1, 0,
            0, 0, 1;
    std::vector<std::string> names = {"A", "B", "C"};
    std::vector<std::string> tied = find_tied_assets0(corr, names);
    // We expect exactly two assets to be identified
    CHECK(tied.size() == 2);
    // Those assets should be A and B
    CHECK(std::find(tied.begin(), tied.end(), "A") != tied.end());
    CHECK(std::find(tied.begin(), tied.end(), "B") != tied.end());
    // C should not be identified
    CHECK(std::find(tied.begin(), tied.end(), "C") == tied.end());


}

TEST_CASE("2. Non-degenerate correlation matrix") {

    Eigen::MatrixXd corr(3, 3);

    corr << 1,   0.2, 0.3,
            0.2, 1,   0.4,
            0.3, 0.4, 1;

    std::vector<std::string> names = {"A", "B", "C"};

    std::vector<std::string> tied = find_tied_assets0(corr, names);

    // No assets should be identified as tied
    CHECK(tied.empty());
}

Overwriting tests/test_engine.cpp


### Run tests


In [ ]:
!g++ -std=c++17 -Wall -Wextra -isystem eigen-3.4.0 -I. -Isrc tests/*.cpp $(ls src/*.cpp | grep -Ev 'main.cpp|correlation_check.cpp') -o tests_run && ./tests_run

[doctest] doctest version is "2.5.3"
[doctest] run with "--help" for options
[doctest] test cases: 2 | 2 passed | 0 failed | 0 skipped
[doctest] assertions: 5 | 5 passed | 0 failed |
[doctest] Status: SUCCESS!


### Run code

In [ ]:
!g++ -std=c++17 -Wall -Wextra -Werror -isystem eigen-3.4.0 src/main.cpp src/CsvReader.cpp src/CollectionSeries.cpp src/PriceSeries.cpp src/Calculations.cpp -o main && ./main

line 13: invalid price in column CCC, row dropped
line 50: empty cell in column EEE, row dropped
line 173: empty cell in column CCC, row dropped
line 179: empty cell in column BBB, row dropped
Duplicate date 2024-09-23, row dropped
Loaded 247 rows, 8 assets
Annualising over 252 trading days
Done. 



In [ ]:
!g++ -std=c++17 -Wall -Wextra -Werror -isystem eigen-3.4.0 src/correlation_check.cpp src/CsvReader.cpp src/CollectionSeries.cpp src/PriceSeries.cpp -o correlation_check && ./correlation_check

line 13: invalid price in column CCC, row dropped
line 50: empty cell in column EEE, row dropped
line 173: empty cell in column CCC, row dropped
line 179: empty cell in column BBB, row dropped
Duplicate date 2024-09-23, row dropped
CCC
DDD
GGG


Checking all the eigenvalues I found one to be of the order 10^-7 which was much larger than the others. In fact, I printed the ratio of the smallest and largest eigenvalues and found that it was smaller than the machine precision I set which was 10^-6. With that I determined that if an eigenvalue was smaller than 10^-6 it would be close enough to assume its value is 0 and thus the correlation matrix is singular. After running the check against the price.csv file I found that assets CCC, DDD, and GGG are all correlated. Its important to note that even though the matrix is not signular in the mathematical sense, it is from a computational sense after we include the fact that the precision of our calculations was only 10^-6 which is larger than the eigenvalue of the matrix.